# Phase 2 — Baseline Evaluation
### Test Qwen 1.5B BEFORE Fine-Tuning

This notebook:
1. Loads Qwen/Qwen1.5-1.8B-Chat base model
2. Runs inference on 200 validation samples
3. Measures SQL accuracy (exact match + execution match)
4. Saves baseline score to baseline_results.json


In [1]:
import os
import json
import torch
import sqlparse
import pandas as pd
from tqdm import tqdm
from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForCausalLM

print("All imports successful!")
print(f"GPU available: {torch.cuda.is_available()}")
print(f"GPU name: {torch.cuda.get_device_name(0)}")


All imports successful!
GPU available: True
GPU name: NVIDIA GeForce RTX 5060 Laptop GPU


## Step 1 — Configuration

In [2]:
CONFIG = {
    "model_name"    : "Qwen/Qwen1.5-1.8B-Chat",
    "data_dir"      : "./data/spider_formatted",
    "output_file"   : "./data/baseline_results.json",
    "num_samples"   : 200,   # evaluate on 200 validation samples
    "max_new_tokens": 128,   # max tokens to generate
    "seed"          : 42,
}

torch.manual_seed(CONFIG["seed"])
print("Config ready!")
print(f"Model       : {CONFIG['model_name']}")
print(f"Evaluating  : {CONFIG['num_samples']} samples")


Config ready!
Model       : Qwen/Qwen1.5-1.8B-Chat
Evaluating  : 200 samples


## Step 2 — Load Validation Dataset

In [3]:
# Load the formatted dataset we saved in Phase 1
dataset = load_from_disk(CONFIG["data_dir"])
val_data = dataset["validation"]

# Take first 200 samples
val_subset = val_data.select(range(CONFIG["num_samples"]))

print(f"Validation samples loaded: {len(val_subset)}")
print(f"\nSample entry:")
print(f"  Question : {val_subset[0]['question']}")
print(f"  Expected : {val_subset[0]['query']}")


Validation samples loaded: 200

Sample entry:
  Question : How many singers do we have?
  Expected : SELECT count(*) FROM singer


## Step 3 — Load Base Model

In [4]:
# Load tokenizer
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    CONFIG["model_name"],
    trust_remote_code=True
)
tokenizer.pad_token = tokenizer.eos_token
print("Tokenizer loaded!")

# Load model in float16 to save VRAM
print("\nLoading model... (this may take 2-3 minutes on first run)")
print("Downloading ~3.5GB on first run, cached after that...")

model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"],
    torch_dtype=torch.float16,
    device_map="cuda",
    trust_remote_code=True
)
model.eval()

print("\nModel loaded successfully!")
print(f"Model device: {next(model.parameters()).device}")

# Check VRAM usage
vram_used = torch.cuda.memory_allocated() / 1024**3
vram_total = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"VRAM used  : {vram_used:.1f} GB / {vram_total:.1f} GB")


Loading tokenizer...


tokenizer_config.json: 0.00B [00:00, ?B/s]

d:\Project\sql_finetune_env\Lib\site-packages\huggingface_hub\file_download.py:147: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\tejaa\.cache\huggingface\hub\models--Qwen--Qwen1.5-1.8B-Chat. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Tokenizer loaded!

Loading model... (this may take 2-3 minutes on first run)


config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.67G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]


Model loaded successfully!
Model device: cuda:0
VRAM used  : 3.8 GB / 8.0 GB


## Step 4 — Run Inference

In [5]:
def generate_sql(question, db_id, model, tokenizer, max_new_tokens=128):
    """Generate SQL query for a given question using the base model."""
    
    prompt = f"""<|im_start|>system
You are an expert SQL assistant that converts natural language questions into accurate SQL queries.<|im_end|>
<|im_start|>user
You are an expert SQL assistant. Given a natural language question and a database name, write the correct SQL query.

Database: {db_id}
Question: {question}

Write only the SQL query, nothing else.<|im_end|>
<|im_start|>assistant
"""
    
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,        # greedy decoding for consistency
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    
    # Decode only the generated part (not the prompt)
    generated = outputs[0][inputs['input_ids'].shape[1]:]
    sql = tokenizer.decode(generated, skip_special_tokens=True).strip()
    
    # Clean up - take only first line if multiple lines
    sql = sql.split('<|im_end|>')[0].strip()
    sql = sql.split('\n\n')[0].strip()
    
    return sql

# Test on one sample first
print("Testing on one sample...")
sample = val_subset[0]
generated_sql = generate_sql(
    sample['question'],
    sample['db_id'],
    model,
    tokenizer
)
print(f"Question : {sample['question']}")
print(f"Expected : {sample['query']}")
print(f"Generated: {generated_sql}")


Testing on one sample...


d:\Project\sql_finetune_env\Lib\site-packages\transformers\generation\configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


Question : How many singers do we have?
Expected : SELECT count(*) FROM singer
Generated: To find the number of singers in the "concert_singer" database, you can use the COUNT function with a WHERE clause to filter the rows where the singer column is not null. Here's the SQL query:


In [6]:
def normalize_sql(sql):
    """Normalize SQL for comparison - lowercase, strip whitespace."""
    sql = sql.lower().strip()
    sql = ' '.join(sql.split())  # normalize whitespace
    # Remove trailing semicolon
    sql = sql.rstrip(';').strip()
    return sql

def exact_match(predicted, expected):
    """Check if predicted SQL exactly matches expected SQL."""
    return normalize_sql(predicted) == normalize_sql(expected)

def token_match(predicted, expected):
    """Partial match - check if key SQL tokens are present."""
    pred_tokens = set(normalize_sql(predicted).split())
    exp_tokens  = set(normalize_sql(expected).split())
    if not exp_tokens:
        return 0.0
    overlap = pred_tokens.intersection(exp_tokens)
    return len(overlap) / len(exp_tokens)

# Test metrics
print("Metrics test:")
print(exact_match("SELECT name FROM city", "select name from city"))  # True
print(token_match("SELECT name FROM city", "select name from city"))   # 1.0


Metrics test:
True
1.0


## Step 5 — Evaluate All 200 Samples

In [7]:
results = []
exact_matches = 0
token_scores  = []

print(f"Evaluating {CONFIG['num_samples']} samples...")
print("This will take 10-20 minutes on RTX 5060\n")

for i, sample in enumerate(tqdm(val_subset, desc="Evaluating")):
    try:
        generated = generate_sql(
            sample['question'],
            sample['db_id'],
            model,
            tokenizer,
            max_new_tokens=CONFIG['max_new_tokens']
        )
        
        em = exact_match(generated, sample['query'])
        tm = token_match(generated, sample['query'])
        
        if em:
            exact_matches += 1
        token_scores.append(tm)
        
        results.append({
            "id"             : i,
            "question"       : sample['question'],
            "db_id"          : sample['db_id'],
            "expected_sql"   : sample['query'],
            "generated_sql"  : generated,
            "exact_match"    : em,
            "token_match"    : round(tm, 4),
        })
        
        # Print progress every 50 samples
        if (i + 1) % 50 == 0:
            current_acc = exact_matches / (i + 1) * 100
            print(f"  [{i+1}/{CONFIG['num_samples']}] Exact match so far: {current_acc:.1f}%")
    
    except Exception as e:
        print(f"  Error on sample {i}: {e}")
        results.append({
            "id"            : i,
            "question"      : sample['question'],
            "db_id"         : sample['db_id'],
            "expected_sql"  : sample['query'],
            "generated_sql" : "ERROR",
            "exact_match"   : False,
            "token_match"   : 0.0,
        })

print("\nEvaluation complete!")


Evaluating 200 samples...
This will take 10-20 minutes on RTX 5060



Evaluating:   0%|          | 0/200 [00:00<?, ?it/s]d:\Project\sql_finetune_env\Lib\site-packages\transformers\generation\configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
Evaluating:  25%|██▌       | 50/200 [02:33<10:02,  4.02s/it]

  [50/200] Exact match so far: 0.0%


Evaluating:  50%|█████     | 100/200 [05:03<06:12,  3.73s/it]

  [100/200] Exact match so far: 0.0%


Evaluating:  75%|███████▌  | 150/200 [09:36<06:19,  7.58s/it]

  [150/200] Exact match so far: 0.0%


Evaluating: 100%|██████████| 200/200 [11:30<00:00,  3.45s/it]

  [200/200] Exact match so far: 0.0%

Evaluation complete!


## Step 6 — Results Summary

In [8]:
# Calculate final scores
exact_match_acc = exact_matches / len(results) * 100
avg_token_match = sum(token_scores) / len(token_scores) * 100

print("=" * 50)
print("  BASELINE EVALUATION RESULTS")
print("=" * 50)
print(f"  Model           : {CONFIG['model_name']}")
print(f"  Samples tested  : {len(results)}")
print(f"  Exact Match Acc : {exact_match_acc:.2f}%")
print(f"  Avg Token Match : {avg_token_match:.2f}%")
print("=" * 50)
print("  (These are your BEFORE scores)")
print("  (After fine-tuning these should improve!)")
print("=" * 50)

# Show some examples
print("\n--- Correct predictions ---")
correct = [r for r in results if r['exact_match']][:3]
for r in correct:
    print(f"  Q: {r['question']}")
    print(f"  Expected : {r['expected_sql']}")
    print(f"  Got      : {r['generated_sql']}")
    print()

print("--- Incorrect predictions ---")
wrong = [r for r in results if not r['exact_match']][:3]
for r in wrong:
    print(f"  Q: {r['question']}")
    print(f"  Expected : {r['expected_sql']}")
    print(f"  Got      : {r['generated_sql']}")
    print()


  BASELINE EVALUATION RESULTS
  Model           : Qwen/Qwen1.5-1.8B-Chat
  Samples tested  : 200
  Exact Match Acc : 0.00%
  Avg Token Match : 36.97%
  (These are your BEFORE scores)
  (After fine-tuning these should improve!)

--- Correct predictions ---
--- Incorrect predictions ---
  Q: How many singers do we have?
  Expected : SELECT count(*) FROM singer
  Got      : To find the number of singers in the "concert_singer" database, you can use the COUNT function with a WHERE clause to filter the rows where the singer column is not null. Here's the SQL query:

  Q: What is the total number of singers?
  Expected : SELECT count(*) FROM singer
  Got      : To find the total number of singers in the "concert_singer" database, you can use the following SQL query:

  Q: Show name, country, age for all singers ordered by age from the oldest to the youngest.
  Expected : SELECT name ,  country ,  age FROM singer ORDER BY age DESC
  Got      : SELECT name, country, age 
FROM concert_singers 


## Step 7 — Save Baseline Results

In [9]:
# Save everything to JSON
baseline_summary = {
    "model"               : CONFIG['model_name'],
    "phase"               : "baseline (before fine-tuning)",
    "samples_evaluated"   : len(results),
    "exact_match_accuracy": round(exact_match_acc, 2),
    "avg_token_match"     : round(avg_token_match, 2),
    "detailed_results"    : results,
}

with open(CONFIG["output_file"], "w") as f:
    json.dump(baseline_summary, f, indent=2)

print(f"Results saved to: {CONFIG['output_file']}")
print("\nPhase 2 Complete!")
print("Your baseline score is locked in.")
print("Next up: Phase 3 - QLoRA Fine-Tuning!")


Results saved to: ./data/baseline_results.json

Phase 2 Complete!
Your baseline score is locked in.
Next up: Phase 3 - QLoRA Fine-Tuning!
